# 172 — Layer Specialization Profiling

Profile what each layer specializes in: prediction impact, information added,
uniqueness, attention vs MLP specialization, and role classification.

## Why This Matters

Layer specialization characterizes what each layer contributes to the model's computation. Understanding layer-level organization — which layers are critical, which are redundant, and how they specialize — is essential for both interpretability and efficient model design.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.layer_specialization_profiling import (
    layer_prediction_impact,
    layer_information_added,
    layer_uniqueness,
    attn_vs_mlp_specialization,
    layer_role_classification,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

In [ ]:
result = layer_prediction_impact(model, tokens)
for p in result['per_layer']:
    changed = 'CHANGED' if p['prediction_changed'] else 'same'
    print(f"Layer {p['layer']}: {changed}  KL={p['kl_divergence']:.4f}  "
          f"delta_norm={p['logit_delta_norm']:.4f}  "
          f"{p['pre_top']}->{p['post_top']}")

In [ ]:
result = layer_information_added(model, tokens)
for p in result['per_layer']:
    print(f"Layer {p['layer']}: new_info(attn={p['attn_new_info']:.4f} mlp={p['mlp_new_info']:.4f})  "
          f"reinforce(attn={p['attn_reinforcement']:.4f} mlp={p['mlp_reinforcement']:.4f})  "
          f"align(attn={p['attn_alignment']:+.3f} mlp={p['mlp_alignment']:+.3f})")

In [ ]:
result = layer_uniqueness(model, tokens)
for p in result['per_layer']:
    bar = '#' * int(p['uniqueness'] * 30)
    print(f"Layer {p['layer']}: unique={p['uniqueness']:.3f}  "
          f"sim={p['mean_similarity_to_others']:.3f}  delta={p['delta_norm']:.4f}  {bar}")

In [ ]:
result = attn_vs_mlp_specialization(model, tokens)
print(f"Target token: {result['target_token']}\n")
for p in result['per_layer']:
    dom = 'ATTN' if p['attn_dominant'] else 'MLP'
    print(f"Layer {p['layer']}: attn={p['attn_logit']:+.4f}  mlp={p['mlp_logit']:+.4f}  "
          f"dominant={dom}  attn_frac={p['attn_logit_fraction']:.3f}")

In [ ]:
result = layer_role_classification(model, tokens)
for p in result['per_layer']:
    print(f"Layer {p['layer']}: {p['role']:15s}  change={p['relative_change']:.4f}  "
          f"alignment={p['pre_delta_alignment']:+.4f}")